
# Faithful Eso-LM-Inspired Implementation Roadmap

This notebook implements a **closer faithful approximation** of:

**Esoteric Language Models: Bridging Autoregressive and Masked Diffusion LLMs**

Key improvements over a simple GPT+masking hybrid:

- Unified permutation-based objective
- Visibility-mask attention
- Bidirectional transformer
- Diffusion-style masking schedule
- Confidence-based iterative decoding
- Shared AR + masked diffusion framework


In [ ]:

import math
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)



# Configuration


In [ ]:

@dataclass
class Config:
    vocab_size = 8000
    seq_len = 128

    n_layers = 6
    n_heads = 6
    dim = 384
    dropout = 0.1

    batch_size = 16
    lr = 3e-4
    max_steps = 25000

    mask_token_id = 0
    pad_token_id = 0

cfg = Config()



# Dataset


In [ ]:
from datasets import load_dataset
import os

if not os.path.exists("tinystories.txt"):
    print("Streaming TinyStories (Safer for Windows)...")
    # Use streaming=True to avoid the pyarrow/multiprocessing crash
    dataset = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
    
    # We'll take 50,000 stories (plenty for a 10M param model)
    num_stories = 50000
    
    with open("tinystories.txt", "w", encoding="utf-8") as f:
        for i, item in enumerate(dataset):
            f.write(item['text'] + "\n\n")
            if (i + 1) >= num_stories:
                break
            if (i + 1) % 5000 == 0:
                print(f"Downloaded {i+1} stories...")
                
    print("Done! Saved to tinystories.txt")


In [ ]:

import sentencepiece as spm
import os

# Train BPE Tokenizer
spm.SentencePieceTrainer.train(
    input='tinystories.txt', 
    model_prefix='tinystories_bpe', 
    vocab_size=cfg.vocab_size,
    user_defined_symbols=['[MASK]'],
    pad_id=3,
    unk_id=0,
    bos_id=1,
    eos_id=2
)

# Load Tokenizer
sp = spm.SentencePieceProcessor()
sp.load('tinystories_bpe.model')

# Dynamically set mask token in config
cfg.mask_token_id = sp.piece_to_id('[MASK]')
cfg.pad_token_id = sp.pad_id()

encode = lambda s: sp.encode_as_ids(s)
decode = lambda x: sp.decode_ids(x)

with open("tinystories.txt", "r", encoding="utf-8") as f:
    text = f.read()

data = torch.tensor(encode(text), dtype=torch.long)

train_split = int(0.9 * len(data))
train_data = data[:train_split]
val_data = data[train_split:]

print("vocab size:", sp.get_piece_size())
print("train tokens:", len(train_data))



# Batch Sampling


In [ ]:

def get_batch(split="train"):
    source = train_data if split == "train" else val_data

    ix = torch.randint(
        len(source) - cfg.seq_len - 1,
        (cfg.batch_size,)
    )

    x = torch.stack([source[i:i+cfg.seq_len] for i in ix])
    x = x.to(device)

    return x



# Cosine Diffusion Mask Schedule

This replaces simplistic linear masking.


In [ ]:

def cosine_mask_rate(t, T, s=0.008):
    f = (t / T + s) / (1 + s)
    return torch.cos(torch.tensor(f) * math.pi / 2) ** 2



# Permutation-Based Corruption

Core Eso-LM idea:
- sample permutation
- reveal subset
- mask remainder


In [ ]:
# corrupt_tokens is now handled by get_esolm_b_diffusion_mask in compute_loss\n


# Rotary Positional Embeddings


In [ ]:

class RotaryEmbedding(nn.Module):

    def __init__(self, dim):
        super().__init__()

        inv_freq = 1.0 / (
            10000 ** (
                torch.arange(0, dim, 2).float() / dim
            )
        )

        self.register_buffer("inv_freq", inv_freq)

    def forward(self, seq_len):

        t = torch.arange(seq_len, device=self.inv_freq.device).type_as(self.inv_freq)

        freqs = torch.einsum("i,j->ij", t, self.inv_freq)

        emb = torch.cat((freqs, freqs), dim=-1)

        return emb.cos(), emb.sin()



# Attention with Visibility Masks

No causal masking.

Attention depends entirely on visible tokens.


In [ ]:
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.dim // cfg.n_heads
        self.qkv = nn.Linear(cfg.dim, cfg.dim * 3)
        self.proj = nn.Linear(cfg.dim, cfg.dim)
        self.dropout = nn.Dropout(cfg.dropout)
        self.rope = RotaryEmbedding(self.head_dim)

    def forward(self, x, attn_bias=None):
        B, L, D = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, L, self.n_heads, self.head_dim).transpose(1,2)
        k = k.view(B, L, self.n_heads, self.head_dim).transpose(1,2)
        v = v.view(B, L, self.n_heads, self.head_dim).transpose(1,2)
        
        cos, sin = self.rope(L)
        cos = cos[None,None,:,:]
        sin = sin[None,None,:,:]
        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        att = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        if attn_bias is not None:
            att = att + attn_bias.to(q.dtype)
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)
        out = att @ v
        out = out.transpose(1,2).contiguous().view(B,L,D)
        return self.proj(out)


# Transformer Block


In [ ]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.dim)
        self.ln2 = nn.LayerNorm(cfg.dim)
        self.attn = Attention()
        self.ff = nn.Sequential(
            nn.Linear(cfg.dim, 4 * cfg.dim),
            nn.GELU(),
            nn.Linear(4 * cfg.dim, cfg.dim),
            nn.Dropout(cfg.dropout)
        )

    def forward(self, x, attn_bias=None):
        x = x + self.attn(self.ln1(x), attn_bias=attn_bias)
        x = x + self.ff(self.ln2(x))
        return x



# Eso-LM Backbone


In [ ]:
class EsoLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.blocks = nn.ModuleList([Block() for _ in range(cfg.n_layers)])
        self.ln = nn.LayerNorm(cfg.dim)
        self.head = nn.Linear(cfg.dim, cfg.vocab_size)

    def forward(self, x, attn_bias=None):
        h = self.token_emb(x)
        for block in self.blocks:
            h = block(h, attn_bias=attn_bias)
        h = self.ln(h)
        return self.head(h)



# Unified Eso-LM Objective

Loss only applied to hidden tokens.


In [ ]:
model = EsoLM().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)

def get_esolm_b_diffusion_mask(is_masked: torch.Tensor):
    B, T = is_masked.shape
    device = is_masked.device
    
    rand_keys = torch.rand(B, T, device=device)
    sort_keys = rand_keys + is_masked.float()
    
    sigma_inv = torch.argsort(torch.argsort(sort_keys, dim=1), dim=1)
    
    sigma_inv_q = sigma_inv.unsqueeze(2)
    sigma_inv_k = sigma_inv.unsqueeze(1)
    allowed = sigma_inv_q >= sigma_inv_k
    
    attn_bias = torch.zeros(B, T, T, device=device, dtype=torch.bfloat16)
    attn_bias.masked_fill_(~allowed, float('-inf'))
    return attn_bias.unsqueeze(1)

def compute_loss(x):
    B, L = x.shape
    t = torch.randint(1, L, (B,), device=x.device)
    mask_prob = t.float() / L
    mask_bcast = mask_prob.unsqueeze(1).expand(B, L)
    is_masked = torch.bernoulli(mask_bcast).bool()
    
    x_corrupt = x.clone()
    x_corrupt[is_masked] = cfg.mask_token_id
    
    if not is_masked.any():
        logits = model(x_corrupt)
        return (logits * 0).sum()
        
    attn_bias = get_esolm_b_diffusion_mask(is_masked)
    logits = model(x_corrupt, attn_bias=attn_bias)
    
    logits_flat = logits.view(B * L, -1)
    targets_flat = x.view(B * L)
    mask_flat = is_masked.view(B * L)
    
    ce = F.cross_entropy(logits_flat, targets_flat, reduction='none')
    loss = ce[mask_flat].mean()
    return loss



# Training Loop


In [ ]:

for step in range(cfg.max_steps):

    model.train()

    x = get_batch()

    loss = compute_loss(x)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())



# Confidence-Based Diffusion Decoding

Important improvement over naive iterative remasking.


In [ ]:
@torch.no_grad()
def generate_diffusion(prompt, steps=32, temperature=0.0, show_steps=False):
    model.eval()
    x = torch.tensor([encode(prompt)], device=device)
    B, L0 = x.shape
    gen_len = cfg.seq_len - L0
    
    mask_tokens = torch.full((1, gen_len), cfg.mask_token_id, device=device)
    x = torch.cat([x, mask_tokens], dim=1)
    
    if show_steps:
        print(f"Init : {decode(x[0].tolist())}")

    for step in range(steps):
        is_masked = (x == cfg.mask_token_id)
        attn_bias = get_esolm_b_diffusion_mask(is_masked)
        
        logits = model(x, attn_bias=attn_bias)
        
        if temperature > 0:
            probs = F.softmax(logits / temperature, dim=-1)
            sampled = torch.distributions.Categorical(probs).sample()
            confidence = probs.gather(-1, sampled.unsqueeze(-1)).squeeze(-1)
        else:
            probs = F.softmax(logits, dim=-1)
            sampled = torch.argmax(logits, dim=-1)
            confidence = probs.max(dim=-1).values

        x[is_masked] = sampled[is_masked]

        remaining_ratio = 1 - (step + 1) / steps
        k = int(remaining_ratio * gen_len)

        if k > 0:
            masked_conf = confidence.clone()
            masked_conf[~is_masked] = 999
            flat = masked_conf.view(-1)
            idx = torch.topk(flat, k, largest=False).indices
            flat_x = x.view(-1)
            flat_x[idx] = cfg.mask_token_id
            x = flat_x.view_as(x)
            
        if show_steps:
            print(f"Step {step+1:02d}: {decode(x[0].tolist())}")

    return decode(x[0].tolist())



# Example Generation


In [ ]:
print(generate_diffusion("Guns\n", steps=100, show_steps=True))
